**About the environment:** This notebook is written for **SageMath**, not plain Python — it uses
`matrix`, `identity_matrix`, `PolynomialRing`, `GF`, `var`, `is_prime`, etc. Run it in a Sage kernel.

# The Han–Monsky algorithm for Hilbert–Kunz functions of diagonal hypersurfaces

This notebook computes the **Hilbert–Kunz function** and **Hilbert–Kunz multiplicity** of a
diagonal hypersurface

$$
R \;=\; \mathbb{F}_p[x_1,\dots,x_s]\big/\big(x_1^{d_1}+\cdots+x_s^{d_s}\big),
$$

using the representation-ring method of **Han and Monsky** [HM93].

### The Hilbert–Kunz function and the Hilbert–Kunz multiplicity

For a local ring $(R,\mathfrak{m})$ of characteristic $p>0$, the
Hilbert–Kunz function is

$$
\mathrm{HK}_R(e) \;=\; \dim_{\mathbb F_p}\, R\big/\mathfrak m^{[p^{e}]},
\qquad
\mathfrak m^{[p^{e}]} = (x_1^{p^{e}},\dots,x_s^{p^{e}}),
$$

and Monsky [Mon83] showed that $\mathrm{HK}_R(e) = c\,p^{e\,\dim R} + O(p^{e(\dim R-1)})$
for a real constant $c=e_{\mathrm{HK}}(R)$, the **Hilbert–Kunz multiplicity**.

### The idea of the algorithm

Han and Monsky [HM93] reduce the computation for a diagonal hypersurface to linear algebra in a
**representation ring** $\Gamma$. The key objects are elements $\lambda_k$ whose multiplication is
encoded by integer matrices. Iterating a "magnification by $p$" structure on these matrices yields
a number (here called $\ell^{\#}$, `lsh` in the code) that controls the eventual shape of
$\mathrm{HK}_R(e)$. Combining $\ell^{\#}$ with two explicitly computed values of the function then
pins down the function for all $e \ge e_0$.

> **References.** The technique was introduced in [HM93]. 
> The theory was later extended by Monsky and Teixeira [MT04, MT05]. In [Yos],
> the matrix representation is introduced.


## 0. Setup and notation

Throughout:

| Symbol in code | Meaning |
|---|---|
| `p` | the prime characteristic |
| `exps` | the exponent vector $(d_1,\dots,d_s)$ of the diagonal hypersurface |
| `dim`  | the Krull dimension $\dim R = s-1$ |
| `n`    | the "level": we work inside $\Lambda_n$, matrices of size $p^n \times p^n$ |
| `lambda_matrix(p,n,k)` | the multiplication matrix of $\lambda_k \in \Lambda_n$ |
| `lsh` ($\ell^{\#}$) | the magnification number controlling the eventual behaviour |
| `mu` ($\mu$) | the eventual **period** of the Hilbert–Kunz function |
| `e0` ($e_0$) | the threshold from which the closed form is valid |

We keep the three layers of the algorithm separate:

1. **The Han–Monsky ring (via matrix representation)** (§1): building the $\lambda_k$ multiplication matrices.
2. **Period search** (§2): the combinatorial part of [HM93, §5] that finds $\mu$ and $e_0$.
3. **Assembly** (§3): solving a small linear system for $e_{\mathrm{HK}}$ and the function.


## 1. The Han–Monsky Ring (via matrix representation)

We briefly introduce the Han–Monsky ring, see [HM93,CR25]. Let $k$ be a fixed field of characteristic $p\geq 0$. Then we call $k$-object any finitely generated $k[T]$-module whose action of $T$ is nilpotent (e.g., $k[T]/(T^n)$ or $k[x,y]/(x^2,y^2)$ where the action $T\cdot f$ is defined by $(x+y)\cdot f$). We call $\Gamma := \Gamma_k$ the Grothendieck ring of the isomorphism classes of the $k$-objects. It follows from the Structure theorem of f.g. modules over a PID that this ring is a free $\mathbb{Z}$-module with basis $k[T]/(T^i)$, and we call $\delta_i$ the isomorphism classes of these elements.

Usually, a different basis is considered: $\lambda_i = (-1)^i(\delta_{i+1}-\delta_i)$. In this section, we have the Han–Monsky ring $\Gamma$, and we can compute the products $\lambda_i\lambda_j$.

We do this via the matrix representation of the elements of $\Gamma$ introduced by Yoshida. It was already observed in [HM93] that the elements $\lbrace \lambda_i\rbrace_{i=0,\dots,p^e-1}$ generate a $\mathbb{Z}$-algebra $\Lambda_e$, and therefore we can express their action by multiplication as an integer matrix over the finite basis $\lbrace \lambda_i\rbrace_{i=0,\dots,p^e-1}$.

This section first builds the multiplication matrix of any $\lambda_k$ inside $\Lambda_e$.

The construction is recursive on the level $e$. We first build the base generators
$\lambda_0,\dots,\lambda_{p-1}$ in $\Lambda_1$ (function `base_lambda`), then lift them to higher
levels using two operators:

- **`theta`** ($\theta$): operator introduced in [MT05], sending $\Lambda_e \to \Lambda_{e+1}$, encoding
  "magnification by $p$";
- **`scale`**: simply scales the matrix, which expresses the inclusion $\Lambda_e\subset \Lambda_{e+1}$. It does so by doing $I_p \otimes M$ (tensoring with the $p\times p$ identity).

Finally `lambda_matrix` writes $k$ in base $p$ as $k = \sum_i a_i\,p^i$ and assembles
$\lambda_k$ as a product of lifted base generators, one per digit.

### Cell-by-cell
First, base_lambda computes the matrix of $\lambda_0,\dots,\lambda_{p-1}$ in $\Lambda_1$.

In [1]:
def base_lambda(p, k):
    r"""
    Multiplication matrix of the generator ``lambda_k`` in ``Lambda_1``.

    These are the building blocks (the elements of ``Lambda_1`` in the notation of
    [HM93]); every ``lambda_k`` in higher levels is assembled from them in
    ``lambda_matrix`` below.

    Parameters
    ----------
    p : prime
        The characteristic.
    k : int
        Index of the generator, with ``0 <= k <= p - 1``.

    Returns
    -------
    A ``p x p`` integer matrix, the multiplication-by-``lambda_k`` matrix in ``Lambda_1``.

    Notes
    -----
    The ``p == 2`` case is handled separately because the general formula below
    assumes ``p`` odd.  Reference: [HM93, Section 2].
    """
    if k > p - 1:
        raise ValueError("k must satisfy 0 <= k <= p - 1.")
    if not p.is_prime():
        raise ValueError("p must be prime.")
    if p == 2:
        return [identity_matrix(2), matrix([[0, 1], [1, 0]])][k]

    # Build the full list of generators, then return the k-th.
    # Each row 'lij' is the indicator of an interval [j0 - i0, i0 + j0], which encodes
    # the structure constants of the representation ring (a Clebsch-Gordan-type rule).
    def indicator_row(i0, j0):
        return [1 if (j0 - i0) <= t <= (i0 + j0) else 0 for t in range(p)]

    rows = []
    for i in range(p):
        for j in range(p):
            if j >= i and i + j <= p - 1:
                rows.append(indicator_row(i, j))
            elif j - i >= 0:
                rows.append(indicator_row(p - 1 - j, p - 1 - i))
            else:
                i0, j0 = j, i
                if i0 + j0 <= p - 1:
                    rows.append(indicator_row(i0, j0))
                else:
                    rows.append(indicator_row(p - 1 - j0, p - 1 - i0))

    generators = [matrix([rows[i + p * k] for i in range(p)]).transpose()
                  for k in range(p)]
    return generators[k]


Now, the next cell has the function $\theta$ introduced in [MT05]: $\lambda_i\mapsto \lambda_{pi}$ if $i$ is even, and $\lambda_{pi-(i-1)}$ if $i$ is odd.

In [2]:
def theta(p, M0):
    r"""
    The operator ``theta : Lambda_n -> Lambda_{n+1}`` of [HM93].

    Given the multiplication matrix ``M0`` of some ``lambda_k`` at level ``n``,
    returns the multiplication matrix of ``theta(lambda_k)`` at level ``n + 1``.
    This is the lift that encodes the genuine "magnification by p" structure of
    the representation ring (as opposed to the trivial lift ``scale`` below).

    Parameters
    ----------
    p : prime
    M0 : matrix
        A ``p^n x p^n`` multiplication matrix at level ``n``.

    Returns
    -------
    A ``p^{n+1} x p^{n+1}`` matrix at level ``n + 1``.
    """
    n = int(log(M0.nrows()) / log(p))

    # Detect which lambda_k this matrix represents: the position of the leading 1
    # in the first column. Its parity drives the placement rule below.
    k = list(M0[0]).index(1)
    parity = k % 2  # 0 (even) or 1 (odd)

    M = zero_matrix(p ** (n + 1), p ** (n + 1))
    for a in range(p):
        for i in range(p ** n):
            for j in range(p ** n):
                # Choose the target row according to the parities of i, j, k.
                if (i % 2 == 0) == (j % 2 == 0):
                    row = p * j + parity * (p - 1) + (-1) ** parity * a
                else:
                    row = p * j + (p - 1) - parity * (p - 1) + (-1) ** (1 - parity) * a
                M[row, p * i + a] = M0[j, i]
    return M
theta(3,base_lambda(3,1))

[0 0 0 1 0 0 0 0 0]
[0 0 0 0 1 0 0 0 0]
[0 0 0 0 0 1 0 0 0]
[1 0 0 0 0 1 1 0 0]
[0 1 0 0 1 0 0 1 0]
[0 0 1 1 0 0 0 0 1]
[0 0 0 1 0 0 0 0 0]
[0 0 0 0 1 0 0 0 0]
[0 0 0 0 0 1 0 0 0]

The function scale simply takes the multiplication matrix of $\lambda_i\in \Lambda_{e_1}$ and outputs the multiplication matrix of the same element in a bigger subring $\Lambda_{e_2}$.

In [3]:
def scale(p, M0):
    r"""
    The trivial lift ``Lambda_n -> Lambda_{n+1}`` given by ``M0 |-> I_p (x) M0``.

    Tensoring on the left with the ``p x p`` identity embeds a level-``n`` matrix
    into level ``n + 1`` without altering its representation-ring content.  Used
    together with ``theta`` to place each base generator at the correct level.
    """
    return identity_matrix(p).tensor_product(M0)


In order to give the multiplication matrix of $\lambda_a$, we need to wirte $a$ in base $p$, so this is what the next function does:

In [4]:
def base_p_digits(p, n, k):
    r"""
    Base-``p`` digits of ``k`` as a length-``n`` list ``[a_0, a_1, ...]``,

    so that ``k = a_0 + a_1 * p + a_2 * p^2 + ...``.  Zero-padded on the right to
    length ``n``.
    """
    digits = []
    while k > 0:
        digits.append(k % p)
        k //= p
    if n > len(digits):
        digits += [0] * (n - len(digits))
    return digits


Finally, the function lambda_matrix(p,n,k) provides the matrix of $\lambda_k$ as an element of $\Lambda_n$ in characteristic $p$.

In [5]:
def lambda_matrix(p, n, k):
    r"""
    Multiplication matrix of ``lambda_k`` in ``Lambda_n``, for any valid ``k``.

    Assembles ``lambda_k`` from the base generators (``base_lambda``) by writing
    ``k`` in base ``p`` and lifting the generator for each digit to level ``n``:
    the ``i``-th digit's generator is pushed up ``i`` times with ``theta`` and
    then ``n - i - 1`` times with ``scale``.  Reference: [HM93, Section 2].

    Parameters
    ----------
    p : prime
    n : int
        The level; the result is a ``p^n x p^n`` matrix.
    k : int
        Index, with ``0 <= k < p^n``.

    Returns
    -------
    A ``p^n x p^n`` integer matrix.
    """
    if k >= p ** n:
        raise ValueError("lambda_%d is not in Lambda_%d." % (k, n))

    digits = base_p_digits(p, n, k)
    M = identity_matrix(p ** n)
    for i in range(n):
        Mi = base_lambda(p, digits[i])
        for _ in range(i):
            Mi = theta(p, Mi)
        for _ in range(n - i - 1):
            Mi = scale(p, Mi)
        M = M * Mi
    return M

lambda_matrix(3,2,7)*lambda_matrix(3,2,4)

[0 0 0 1 1 1 0 0 0]
[0 0 0 1 3 1 0 0 0]
[0 0 0 1 1 1 0 0 0]
[1 1 1 1 1 1 1 1 1]
[1 3 1 1 3 1 1 3 1]
[1 1 1 1 1 1 1 1 1]
[0 0 0 1 1 1 0 0 0]
[0 0 0 1 3 1 0 0 0]
[0 0 0 1 1 1 0 0 0]

The functions above ask you to supply the level $n$ explicitly, but for a product
$\lambda_a \cdot \lambda_b$ there is a natural choice of $n$: you need the smallest $n$ with
$p^{n} > \max(a,b)$, and any larger level gives the same answer, only embedded in a bigger matrix.

In [6]:
p = 3
e = 2
i = 3
lambda_matrix(p,e,i)

[0 0 0 1 0 0 0 0 0]
[0 0 0 0 1 0 0 0 0]
[0 0 0 0 0 1 0 0 0]
[1 0 0 0 0 1 1 0 0]
[0 1 0 0 1 0 0 1 0]
[0 0 1 1 0 0 0 0 1]
[0 0 0 1 0 0 0 0 0]
[0 0 0 0 1 0 0 0 0]
[0 0 0 0 0 1 0 0 0]

## 1b. Computing in the ring: `Lambda` elements

This section wraps the matrix machinery in a small element class, `Lambda`, so that the
generators behave like elements of the representation ring $\Gamma$ of [HM93] that you can
multiply with the ordinary `*`:

```
G = HanMonsky(3)      # the ring in characteristic 3
G(4) * G(6)           # -> a Lambda-combination, level inferred to be 2
```

**How a product is read back.** The matrix of $\lambda_k$ at level $n$ is *multiplication by
$\lambda_k$* written in the $\lambda$-basis $\{\lambda_0,\dots,\lambda_{p^n-1}\}$ (recall
$\lambda_0 = 1$ is the identity). Column $j$ of that matrix is therefore the coefficient vector
of $\lambda_k \cdot \lambda_j$. In particular **column $0$** is the coefficient vector of
$\lambda_k \cdot \lambda_0 = \lambda_k$ itself, and the matrix product
$M(\lambda_a)\,M(\lambda_b)$ is the matrix of $\lambda_a\lambda_b$, whose column $0$ gives the
product re-expressed as $\sum_k c_k \lambda_k$. That is how `Lambda` prints its results and how
it supports addition. Reference: [HM93, Section 2].


In [7]:
def _min_level(p, indices):
    r"""
    Smallest level ``n`` with ``p^n > max(indices)`` (and at least 1).

    This is the minimal level whose basis ``{lambda_0, ..., lambda_{p^n - 1}}``
    contains every requested generator.
    """
    m = max(indices) if indices else 0
    n = 1
    while p ** n <= m:
        n += 1
    return n


class Lambda:
    r"""
    An element of the Han-Monsky representation ring ``Gamma`` in characteristic ``p``.

    An element is stored as a dictionary ``{index: coefficient}`` of ``lambda``-basis
    generators, e.g. ``lambda_2 + 3*lambda_5`` is ``{2: 1, 5: 3}``.  The level is never
    stored: it is inferred from the indices whenever a matrix is needed (see ``_min_level``).

    Build single generators with the ``HanMonsky`` factory below, e.g. ``G = HanMonsky(3)``
    then ``G(4)``.  Multiply and add with the ordinary ``*`` and ``+``.

    Reference: [HM93, Section 2].
    """

    def __init__(self, p, coeffs):
        if not p.is_prime():
            raise ValueError("p must be prime.")
        self.p = p
        # Drop zero coefficients so equality and printing are canonical.
        self.coeffs = {k: c for k, c in coeffs.items() if c != 0}

    # --- the level is implicit: inferred from the indices in play ---
    def level(self):
        """Minimal level whose basis contains all generators of this element."""
        return _min_level(self.p, list(self.coeffs) or [0])

    def matrix(self, level=None):
        r"""
        Multiplication matrix of this element, in the lambda-basis.

        By default uses the minimal level; pass ``level`` to embed in a larger one
        (the matrix grows but represents the same element).
        """
        n = self.level() if level is None else level
        if any(k >= self.p ** n for k in self.coeffs):
            raise ValueError("level %d too small for indices %s"
                             % (n, sorted(self.coeffs)))
        M = zero_matrix(self.p ** n, self.p ** n)
        for k, c in self.coeffs.items():
            M += c * lambda_matrix(self.p, n, k)
        return M

    # --- arithmetic ---
    def __mul__(self, other):
        if self.p != other.p:
            raise ValueError("characteristic mismatch: %d vs %d" % (self.p, other.p))
        # Work at the level holding both factors' indices; the product matrix's
        # column 0 is the product re-expressed in the lambda-basis.
        n = _min_level(self.p, list(self.coeffs) + list(other.coeffs) or [0])
        product = self.matrix(n) * other.matrix(n)
        column0 = product.column(0)
        return Lambda(self.p, {k: column0[k] for k in range(len(column0))})

    def __add__(self, other):
        if self.p != other.p:
            raise ValueError("characteristic mismatch: %d vs %d" % (self.p, other.p))
        out = dict(self.coeffs)
        for k, c in other.coeffs.items():
            out[k] = out.get(k, 0) + c
        return Lambda(self.p, out)

    def __eq__(self, other):
        return (isinstance(other, Lambda)
                and self.p == other.p and self.coeffs == other.coeffs)

    # --- display ---
    def __repr__(self):
        if not self.coeffs:
            return "0"
        parts = []
        for k in sorted(self.coeffs):
            c = self.coeffs[k]
            parts.append(("lambda_%d" % k) if c == 1
                         else ("%s*lambda_%d" % (c, k)))
        return " + ".join(parts)


def HanMonsky(p):
    r"""
    The Han-Monsky ring in characteristic ``p``.

    Returns a function ``G`` such that ``G(k)`` is the generator ``lambda_k`` as a
    ``Lambda`` element.  Use ``*`` and ``+`` to compute:

        G = HanMonsky(3)
        G(4) * G(6)        # level inferred to be 2
    """
    p = Integer(p)
    return lambda k: Lambda(p, {Integer(k): Integer(1)})


In [8]:
# Computing in the ring, with the level inferred automatically.
G = HanMonsky(3)

print("lambda_4 * lambda_6 =", G(4) * G(6))      # inferred level 2 (since 3^2 = 9 > 6)
print("inferred level:", (G(4) * G(6)).level())

# Sanity check: the result, turned back into a matrix, must equal the
# direct matrix product of the two factors at the same level.
n = _min_level(3, [4, 6])
(G(4) * G(6)).matrix(2) == lambda_matrix(3, n, 4) * lambda_matrix(3, n, 6)
G(7)*G(4)

lambda_4 * lambda_6 = lambda_4
inferred level: 2


lambda_3 + lambda_4 + lambda_5

## 2. The Han–Monsky algorithm

_add explanation from paper_

# 2.1. Finding the period $\mu$ and threshold $e_0$

This cells consist of the actual Han–Monsky algorithm, corresponding to [HM93, §5].

Given the exponents, set $\beta = (1/d_1,\dots,1/d_s)$. The algorithm:

1. **Strips the prime from the denominators.** Compute the largest $e_0$ with $p^{e_0}\mid d_i$
   for the relevant $d_i$, and replace $\beta$ by $p^{e_0}\beta$ so that no denominator is divisible
   by $p$.
2. **Searches for the smallest period $\mu$.** It looks for the least $\mu \ge 1$ such that the
   vector $z' = (p^\mu \beta \bmod 1)$ lands in the same equivalence class (under the sign group
   $\{\pm1\}^s$ with product $+1$ or $-1$) as the original $z$, with a matching parity condition on
   $\sum r_i$. By [HM93, Theorem 5.2], when the hypersurface is an $A_n$ singularity (so
   $n+1 = c\,p^{e_0}$ with $\gcd(c,p)=1$), this $\mu$ is the least exponent with $p^\mu \equiv \pm1 \pmod c$.

The function returns $(\mu, e_0, \beta_{\text{new}})$.


In [9]:
def find_period(p, exps, ideal, verbose=False,talk=False):
    r"""
    Find the period ``mu`` and threshold ``e0`` for the Han-Monsky algorithm.

    Implements the search of [HM93, Section 5]: it normalises ``beta = (1/d_i)``
    so that no denominator is divisible by ``p`` (this defines ``e0``), then finds
    the least ``mu`` for which the magnified vector ``p^mu * beta`` returns to the
    same sign-equivalence class as ``beta``, with a matching parity condition.

    For an ``A_n`` singularity, where ``n + 1 = c * p^{e0}`` with ``gcd(c, p) = 1``,
    this ``mu`` is the least exponent with ``p^mu == +/- 1 (mod c)``
    (see [HM93, Theorem 5.2]).

    Parameters
    ----------
    p : prime
    exps : list of int (d_1,...,d_s)
        Corresponds to hypersurface x_1^d_1+...+x_s^d_s.
    ideal : list of int (a_1,...,a_s)
        Corresponds to m-primary ideal (x_1^a_1,...,x_s^a_s)
    verbose : bool
        If True, print intermediate quantities.

    Returns
    -------
    (mu, e0, new_beta) : (int, int, vector)
    """
    import itertools

    beta = vector([ideal[i] / exps[i] for i in range(len(exps))])

    # Step 1: strip p from the denominators to find e0.
    e0 = 0
    for d in exps:
        while d % p ** (e0 + 1) == 0:
            e0 += 1
    new_beta = p ** e0 * beta
    fracs = [(numerator(x), denominator(x)) for x in new_beta]
    r = vector([int(x) for x in new_beta])

    # The two sign-equivalence classes (product of signs +1 vs -1) of [HM93, Sec 5].
    units = list(itertools.product([1, -1], repeat=len(exps)))
    class_z = [[u[i] * fracs[i][0] % fracs[i][1] for i in range(len(exps))]
               for u in units if prod(u) == 1]
    class_z_star = [[u[i] * fracs[i][0] % fracs[i][1] for i in range(len(exps))]
                    for u in units if prod(u) == -1]

    # Step 2: try mu = 1, 2, ... up to the a-priori bound max_mu.
    max_mu = lcm([euler_phi(den) for (_, den) in fracs])
    if verbose:
        print("Maximum mu:", max_mu)

    def matches(mu):
        z_prime = [p ** mu * num % den for (num, den) in fracs]
        R = [int(x) for x in p ** mu * new_beta]
        same_parity = (sum(R) % 2 == sum(r) % 2)
        in_z = z_prime in class_z
        in_z_star = z_prime in class_z_star
        if verbose:
            print("mu=%d  z'=%s  z~z'=%s  z*~z'=%s  parity=%s"
                  % (mu, z_prime, in_z, in_z_star, same_parity))
        return (in_z and same_parity) or (in_z_star and not same_parity)

    mu = 1
    while not matches(mu) and mu < max_mu:
        mu += 1
    if talk:
        return 'Period mu = '+str(mu)+', e0 = '+str(e0)
    return mu, e0, new_beta
find_period(5,[3,3,3,3],[1,1,1,2],talk=True)

'Period mu = 1, e0 = 0'

## 3. Assembling the Hilbert–Kunz function

With the matrix machinery (§1) and the period (§2) in hand, the final step computes:

- $\ell^{\#}$ (`lsh`): the $(0,0)$ entry of a product of $\lambda_{t_i}$ matrices at level $\mu$,
  where the vector $t$ is read off from $p^\mu z$. This is the magnification number.
- Two values of the Hilbert–Kunz function, $\mathrm{HK}(e_0)$ and $\mathrm{HK}(e_0+\mu)$, computed
  **by brute force** as $\mathbb F_p$-vector-space dimensions of the relevant quotient. (They can
  be obtained from matrices too, but the matrices grow fast and there is no practical gain.)

These satisfy a $2\times 2$ linear system
$$
\mathrm{HK}(e_0) = c\,p^{\dim\cdot e_0} - y,\qquad
\mathrm{HK}(e_0+\mu) = c\,p^{\dim(e_0+\mu)} - y\,\ell^{\#},
$$
whose solution gives the multiplicity $c = e_{\mathrm{HK}}$ and the correction term, hence the full
function for $e\ge e_0$.


In [10]:
def _hk_value(p, exps, ideal, e):
    r"""
    Brute-force value of the Hilbert-Kunz function at level ``e``:

        dim_{F_p}  F_p[x]/( sum x_i^{d_i},  x_1^{p^e}, ..., x_s^{p^e} ).

    Computed directly as an ``F_p``-vector-space dimension.  Reference: [HM93].
    """
    if ideal == None:
        ideal = len(exps)*[1]
    s = len(exps)
    variables = var(['x%d' % i for i in range(s)])
    ring = PolynomialRing(GF(p), variables)
    hypersurface = sum(var('x%d' % i) ** exps[i] for i in range(s))
    frobenius = [var('x%d' % i) ** (ideal[i]*p ** e) for i in range(s)]
    return ring.ideal([hypersurface] + frobenius).vector_space_dimension()

In [11]:
p = 7
e = 1
float(_hk_value(p,[3,3,3,3],[1,1,1,2],e)/p^(3*e))

2.868804664723032

In [12]:
def eHK_diagonal(p, exps, ideal = None, verbose=False,
                  return_HK_function=False, full_HK_function=False):
    r"""
    Hilbert-Kunz multiplicity (or function) of a diagonal hypersurface via [HM93].

        R = F_p[x_1,...,x_s] / (x_1^{d_1} + ... + x_s^{d_s})

    The computation needs three ingredients: two explicit values of the Hilbert-Kunz
    function (computed by ``_hk_value``) and the magnification number ``lsh`` = ``l#``
    (computed from the level-``mu`` lambda matrices).

    Parameters
    ----------
    p : prime
    exps : list of int (d_1,...,d_s)
        Corresponds to hypersurface x_1^d_1+...+x_s^d_s.
    ideal : list of int (a_1,...,a_s)
        Corresponds to m-primary ideal (x_1^a_1,...,x_s^a_s)
    verbose : bool
        Print intermediate quantities.
    return_HK_function : bool
        If True, return a string giving the Hilbert-Kunz *function* rather than just
        the multiplicity.  When ``mu > 1`` the identity only holds along the residue
        class ``e mod mu``, so the returned expression is written e.g. as
        ``HK_(3e) = ...``.  See note below.
    full_HK_function : bool
        If True (and ``return_HK_function`` True), return all ``mu`` pieces of the
        function, one per residue class mod ``mu``.

    Returns
    -------
    The Hilbert-Kunz multiplicity (a rational number), or a string describing the
    Hilbert-Kunz function.
    """
    if ideal == None:
        ideal = len(exps)*[1]
    if verbose:
        print("Original beta:", vector([1 / d for d in exps]))

    mu, e0, beta = find_period(p, exps, ideal)
    if verbose:
        print("mu =", mu, ", e0 =", e0, ", new beta =", beta)

    dim = len(exps) - 1
    r = vector([floor(x) for x in beta])
    z = beta - r
    t = vector([floor(x) for x in p ** mu * z])

    if verbose:
        print("sum(r) odd:", sum(r) % 2 != 0, "(sum r =", sum(r), ")")
    if sum(r) % 2 != 0:
        t[0] = p ** mu - 1 - t[0]

    # l# (lsh): the magnification number, from the level-mu lambda matrices.
    lsh = prod(lambda_matrix(p, mu, ti) for ti in t)[0, 0]
    if verbose:
        print("t-vector =", t, " so l# =", lsh)

    # Two explicit Hilbert-Kunz values, then solve the 2x2 linear system.
    HKe0 = _hk_value(p, exps, ideal, e0)
    HKe0mu = _hk_value(p, exps, ideal, e0 + mu)

    eHK = (HKe0mu - HKe0 * lsh) / ((p ** (mu * dim) - lsh) * p ** (e0 * dim))

    if not return_HK_function:
        return eHK

    # --- function-valued output ---
    var('e')
    if not full_HK_function:
        Delta = eHK * p ** (e0 * dim) - HKe0
        return ('HK_(%se) = %s - %s  for e >= %s'
                % (mu, eHK * p ** (dim * (mu * e)), Delta * lsh ** e, e0))

    Deltas = [eHK * p ** ((e0 + k) * dim) - _hk_value(p, exps, ideal, e0 + k)
              for k in range(mu)]
    if len(Deltas) == 1:
        pieces = "%s for all e" % Deltas[0]
    else:
        var('m')
        pieces = ", ".join("%s for e = %s" % (Deltas[i], mu * m + i)
                           for i in range(len(Deltas)))
    if mu>1:
        if e0>0:
            return ('%s - Delta_(e-%s) * %s^floor((e-%s)/%s)  for e >= %s,  where Delta_e is %s'
                    % (eHK * p ** (dim * e), e0, lsh, e0, mu, e0, pieces))
        else:
            return ('%s - Delta_e * %s^floor(e/%s)  for e >= %s,  where Delta_e is %s'
                % (eHK * p ** (dim * e), lsh, mu, e0, pieces))
    else:
        if e0>0:
            return ('%s - Delta_(e-%s) * %s^(e-%s)  for e >= %s,  where Delta_e is %s'
                    % (eHK * p ** (dim * e), e0, lsh, e0, e0, pieces))
        else:
            return ('%s - Delta_e * %s^e  for e >= %s,  where Delta_e is %s'
                % (eHK * p ** (dim * e), lsh, e0, pieces))

In [13]:
def HK_function_diagonal(p, exps, ideal = None, verbose=False):
    """Hilbert-Kunz function (one residue class) of the diagonal hypersurface."""
    if ideal == None:
        ideal = len(exps)*[1]
    return eHK_diagonal(p, exps, ideal, verbose, return_HK_function=True)

def full_HK_function_diagonal(p, exps, ideal = None, verbose=False):
    """Full Hilbert-Kunz function (all residue classes) of the diagonal hypersurface."""
    if ideal == None:
        ideal = len(exps)*[1]
    return eHK_diagonal(p, exps, ideal, verbose,
                        return_HK_function=True, full_HK_function=True)

def F_signature_diagonal(p, exps, verbose=False,
                  return_HK_function=False, full_HK_function=False):
    s = len(exps)
    HS_mult = min(exps)
    I_delta = (s-1)*[1]+[exps[s-1]-1]
    return HS_mult-eHK_diagonal(p,exps,I_delta,verbose)

## 4. Examples

Three worked examples.

First, from [HM93]

In [14]:
# e_HK of F_5[x,y,z,t] / (x^4 + y^4 + z^4 + t^3)
print("e_HK(F_5[x,y,z,t]/(x^4+y^4+z^4+t^4)) =")
eHK_diagonal(5, [4, 4, 4, 4])

e_HK(F_5[x,y,z,t]/(x^4+y^4+z^4+t^4)) =


168/61

In [15]:
# Hilbert-Kunz function of F_5[x,y,z,t] / (x^7 + y^7 + z^7 + t^7)
print("HK_e(F_5[x,y,z,t]/(x^4+y^4+z^4+t^4)) =")
print(full_HK_function_diagonal(5, [4, 4, 4, 4]))

HK_e(F_5[x,y,z,t]/(x^4+y^4+z^4+t^4)) =
168/61*5^(3*e) - Delta_e * 3^e  for e >= 0,  where Delta_e is 107/61 for all e


Example from [...]

In [ ]:
p = 7
# F-signature of F_5[x,y,z,t] / (x^3 + y^3 + z^3 + t^3)
print("s(F_%s[x,y,z,t]/(x^3+y^3+z^3+t^3)) = "%p + str(F_signature_diagonal(p, [3, 3, 3, 3])))
'Does it coincide with the formula in Theorem 7.1 in [CSZT]? ' + str(F_signature_diagonal(p, [3, 3, 3, 3]) == 3*p*(p-1)*(p+1)/(8*(3*p^3-p+2*(-1)^(p%3))))

s(F_7[x,y,z,t]/(x^3+y^3+z^3+t^3)) = 21/170


'It coincides with the formula in Theorem 7.1 in [CSZT]: True'

Example from [Yos]

In [17]:
#print("HK_e(F_5[x,y,z,t]/(x^4+y^4+z^4+t^4)) =")
print(full_HK_function_diagonal(3, [6, 6, 6, 5]))

459/121*3^(3*e) - Delta_(e-1) * 3^floor((e-1)/2)  for e >= 1,  where Delta_e is 2592/121 for e = 2*m, 4644/121 for e = 2*m + 1


In [18]:
p = 3
Delta = [2592/121,4644/121]
e = 4
R.<x,y,z,t> = PolynomialRing(GF(p))
I = R.ideal([x^6+y^6+z^6+t^5,x^(p^e),y^(p^e),z^(p^e),t^(p^e)])
I.vector_space_dimension(),459/121*3^(3*e) - Delta[(e-1)%2] * 3^floor((e-1)/2)  ,e

(2015847, 2015847, 4)

## 5. References

- **[HM93]** C. Han and P. Monsky. *Some surprising Hilbert–Kunz functions*,
  Mathematische Zeitschrift **214** (1993), no. 1, 119–135.
  DOI: [10.1007/BF02572395](https://doi.org/10.1007/BF02572395).

- **[CSTZ]** Caminata, A., Shideler, S., Tucker, K., & Zerman, F. *F-signature functions of diagonal hypersurfaces*, arXiv preprint arXiv:2403.12863, (2024)

- **[Han91]** C. Han. *The Hilbert–Kunz function of a diagonal hypersurface*,
  Ph.D. thesis, Brandeis University, 1991.

- **[Mon83]** P. Monsky. *The Hilbert–Kunz function*,
  Mathematische Annalen **263** (1983), 43–49.

- **[MT04]** P. Monsky and P. Teixeira. *p-Fractals and power series — I. Some 2-variable results*,
  Journal of Algebra **280** (2004), 505–536.
  DOI: [10.1016/j.jalgebra.2004.05.016](https://doi.org/10.1016/j.jalgebra.2004.05.016).

- **[MT05]** P. Monsky and P. Teixeira. *p-Fractals and power series — II. Some applications to
  Hilbert–Kunz theory*, Journal of Algebra **304** (2006), 237–255 (arXiv:math/0506584, 2005).

- **[Yos]** Yoshida, Kei-Ichi. *Small Hilbert-Kunz multiplicity and $(A_1)$-type singularity*,
  Proceedings of the 4th Japan-Vietnam Joint Seminar on Commutative Algebra by and for Young Mathematicians, Meiji University, Japan (2009).
